# Eventos de violencia organizada (UCDP GED)

El dataset UCDP GED documenta eventos de violencia organizada en el mundo, incluyendo detalles temporales, geográficos y del tipo de conflicto. Este proyecto busca explorar patrones de violencia, tendencias temporales, distribución geográfica y posibles indicadores predictivos.

El mismo puede descargarse desde https://ucdp.uu.se/downloads/index.html#ged_global

In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv("dataset/GEDEvent_v25_1.csv", low_memory=False)
data.head(5)

,id,relid,year,active_year,code_status,type_of_violence,conflict_dset_id,conflict_new_id,conflict_name,dyad_dset_id,...,date_end,deaths_a,deaths_b,deaths_civilians,deaths_unknown,best,high,low,gwnoa,gwnob
0,244657,IRQ-2017-1-524-322,2017,True,Clear,1,259,259,Iraq: Government,524,...,2017-07-31 00:00:00.000,0,4,0,2,6,6,6,645,NaN
1,412700,IRQ-2021-1-524-145,2021,True,Clear,1,259,259,Iraq: Government,524,...,2021-08-26 00:00:00.000,13,1,141,28,183,184,171,645,NaN
2,413023,IRQ-2021-1-524-143,2021,True,Clear,1,259,259,Iraq: Government,524,...,2021-08-28 00:00:00.000,0,2,0,0,2,3,0,645,NaN
3,412909,IRQ-2021-1-524-144,2021,True,Clear,1,259,259,Iraq: Government,524,...,2021-08-29 00:00:00.000,0,0,10,0,10,10,9,645,NaN
4,132140,AFG-1989-1-411-2,1989,True,Clear,1,333,333,Afghanistan: Government,724,...,1989-01-13 00:00:00.000,6,0,0,0,6,6,6,700,NaN


## Descripción general del dataset

In [13]:
# ─────────────────────────────────────────────
# RESUMEN GENERAL — UCDP GED
# ─────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

num_rows, num_cols = data.shape

# Rango temporal
year_col = next((c for c in data.columns if 'year' in c.lower()), None)
year_range = f"{int(data[year_col].min())} - {int(data[year_col].max())}" if year_col else "No disponible"

# ── Clasificación semántica de columnas ──────────────────────────────────
# IDs y códigos internos (no analíticos)
ID_COLS   = ['id', 'relid', 'conflict_dset_id', 'conflict_new_id',
             'dyad_dset_id', 'dyad_new_id', 'side_a_dset_id', 'side_a_new_id',
             'side_b_dset_id', 'side_b_new_id', 'priogrid_gid', 'country_id',
             'code_status', 'gwnoa', 'gwnob']

# Fechas
DATE_COLS = ['date_start', 'date_end', 'source_date']

# Texto libre / fuente (no analítico)
TEXT_COLS = ['source_article', 'source_office', 'source_headline',
             'source_original', 'where_coordinates', 'where_description',
             'adm_1', 'adm_2', 'geom_wkt']

# Columnas categóricas analíticas (útiles para agrupar/filtrar)
CATEGORICAL_COLS = ['conflict_name', 'dyad_name', 'side_a', 'side_b',
                    'country', 'region', 'type_of_violence', 'where_prec',
                    'event_clarity', 'date_prec', 'number_of_sources']

# Columnas numéricas analíticas
NUMERIC_COLS = ['year', 'latitude', 'longitude',
                'deaths_a', 'deaths_b', 'deaths_civilians', 'deaths_unknown',
                'best', 'high', 'low']

# Filtrar solo las que existen en el dataset cargado
id_cols          = [c for c in ID_COLS          if c in data.columns]
date_cols        = [c for c in DATE_COLS        if c in data.columns]
text_cols        = [c for c in TEXT_COLS        if c in data.columns]
categorical_vars = [c for c in CATEGORICAL_COLS if c in data.columns]
numeric_vars     = [c for c in NUMERIC_COLS     if c in data.columns]

# Columnas con alto porcentaje de nulos (> 40%)
null_pct       = data.isnull().mean().sort_values(ascending=False)
high_null_cols = null_pct[null_pct > 0.4]

print("\n══════════════════════════════════════════")
print("  RESUMEN DEL DATASET — UCDP GED")
print("══════════════════════════════════════════")
print(f"  Filas              : {num_rows:,}")
print(f"  Columnas           : {num_cols}")
print(f"  Columnas tipo ID   : {len(id_cols)}")
print(f"  Columnas de fecha  : {len(date_cols)}")
print(f"  Columnas texto/src : {len(text_cols)}")
print(f"  Rango de años      : {year_range}")
print()
print("── Variables categóricas ────")
for v in categorical_vars:
    n_unique = data[v].nunique()
    print(f"   • {v:<30} ({n_unique:,} valores únicos)")
print()
print("── Variables numéricas ──────")
for v in numeric_vars:
    print(f"   • {v}")
print()
print("── Features con > 40% nulos ──────────────")
if high_null_cols.empty:
    print("   Ninguna")
else:
    for col, pct in high_null_cols.items():
        print(f"   • {col:<40} {pct*100:.1f}%")
print("══════════════════════════════════════════")


══════════════════════════════════════════
  RESUMEN DEL DATASET — UCDP GED
══════════════════════════════════════════
  Filas              : 385,918
  Columnas           : 49
  Columnas tipo ID   : 15
  Columnas de fecha  : 3
  Columnas texto/src : 9
  Rango de años      : 1989 - 2024

── Variables categóricas ────
   • conflict_name                  (1,529 valores únicos)
   • dyad_name                      (1,771 valores únicos)
   • side_a                         (966 valores únicos)
   • side_b                         (959 valores únicos)
   • country                        (124 valores únicos)
   • region                         (5 valores únicos)
   • type_of_violence               (3 valores únicos)
   • where_prec                     (7 valores únicos)
   • event_clarity                  (2 valores únicos)
   • date_prec                      (5 valores únicos)
   • number_of_sources              (54 valores únicos)

── Variables numéricas ──────
   • year
   • latitude
   • l

## Hallazgos y estadísticas clave

In [4]:
# ─────────────────────────────────────────────
# 1. DISTRIBUCIÓN POR TIPO DE VIOLENCIA
# ─────────────────────────────────────────────
# type_of_violence: 1=state-based, 2=non-state, 3=one-sided
type_labels = {1: 'Violencia estatal', 2: 'No estatal', 3: 'Unilateral'}

if 'type_of_violence' in data.columns:
    tov = data['type_of_violence'].value_counts().sort_index()
    print("\n── Tipo de violencia ─────────────────────")
    for k, v in tov.items():
        label = type_labels.get(k, str(k))
        print(f"   {label:<25} {v:>8,}  ({v/len(data)*100:.1f}%)")


── Tipo de violencia ─────────────────────
   Violencia estatal          271,331  (70.3%)
   No estatal                  54,982  (14.2%)
   Unilateral                  59,605  (15.4%)


In [5]:
# ─────────────────────────────────────────────
# 2. PAÍSES Y REGIONES CON MÁS EVENTOS
# ─────────────────────────────────────────────
TOP_N = 10

if 'country' in data.columns:
    top_countries = data['country'].value_counts().head(TOP_N)
    print(f"\n── Top {TOP_N} países por número de eventos ──")
    for country, count in top_countries.items():
        print(f"   {country:<35} {count:>7,}")

if 'region' in data.columns:
    top_regions = data['region'].value_counts().head(TOP_N)
    print(f"\n── Top {TOP_N} regiones por número de eventos ─")
    for region, count in top_regions.items():
        print(f"   {region:<35} {count:>7,}")


── Top 10 países por número de eventos ──
   Syria                                87,861
   Afghanistan                          42,220
   Ukraine                              31,547
   Mexico                               21,550
   India                                17,997
   Colombia                             14,620
   Iraq                                  9,438
   Bosnia-Herzegovina                    9,340
   DR Congo (Zaire)                      8,901
   Myanmar (Burma)                       8,476

── Top 10 regiones por número de eventos ─
   Middle East                         122,215
   Asia                                 97,879
   Africa                               68,354
   Americas                             48,884
   Europe                               48,586


In [6]:
# ─────────────────────────────────────────────
# 3. BAJAS (MUERTES) — estadísticas descriptivas
# ─────────────────────────────────────────────
death_cols = [c for c in data.columns if 'death' in c.lower() or 'best' in c.lower()]

if death_cols:
    print("\n── Estadísticas de bajas ─────────────────")
    print(data[death_cols].describe().round(1).to_string())
    print()

# Total de muertes estimadas (columna 'best' es la estimación central del UCDP)
if 'best' in data.columns:
    total_deaths   = data['best'].sum()
    median_deaths  = data['best'].median()
    max_event      = data.loc[data['best'].idxmax()]
    print(f"   Total muertes estimadas (best)  : {total_deaths:>12,.0f}")
    print(f"   Mediana por evento              : {median_deaths:>12.1f}")
    if 'conflict_name' in data.columns:
        print(f"   Evento con más muertes         : {max_event['conflict_name']} ({int(max_event['best']):,} bajas)")


── Estadísticas de bajas ─────────────────
       deaths_a  deaths_b  deaths_civilians  deaths_unknown      best
count  385918.0  385918.0          385918.0        385918.0  385918.0
mean        2.5       2.2               3.8             1.7      10.3
std       254.7      53.8             178.4            99.6     347.7
min         0.0       0.0               0.0             0.0       0.0
25%         0.0       0.0               0.0             0.0       1.0
50%         0.0       0.0               0.0             0.0       2.0
75%         0.0       1.0               1.0             0.0       4.0
max    121848.0   19874.0           40000.0         48183.0  121848.0

   Total muertes estimadas (best)  :    3,957,143
   Mediana por evento              :          2.0
   Evento con más muertes         : Ethiopia: Government (121,848 bajas)


In [7]:
# ─────────────────────────────────────────────
# 4. TENDENCIA TEMPORAL — eventos por año
# ─────────────────────────────────────────────
if year_col:
    events_per_year = data[year_col].value_counts().sort_index()
    peak_year       = int(events_per_year.idxmax())
    peak_events     = int(events_per_year.max())
    quiet_year      = int(events_per_year.idxmin())
    quiet_events    = int(events_per_year.min())

    print("\n── Tendencia temporal ────────────────────")
    print(f"   Año con más eventos  : {peak_year}  ({peak_events:,} eventos)")
    print(f"   Año con menos eventos: {quiet_year} ({quiet_events:,} eventos)")
    print()
    print(events_per_year.rename('eventos').to_frame().to_string())


── Tendencia temporal ────────────────────
   Año con más eventos  : 2024  (28,816 eventos)
   Año con menos eventos: 1989 (2,624 eventos)

      eventos
year         
1989     2624
1990     3202
1991     2870
1992     7333
1993     6828
1994     7569
1995     4697
1996     3294
1997     3007
1998     3787
1999     4412
2000     5725
2001     4363
2002     7467
2003     6423
2004     7533
2005     6125
2006     5604
2007     6927
2008     6715
2009     7146
2010     7885
2011     7715
2012    18540
2013    24521
2014    25940
2015    19875
2016    17374
2017    16304
2018    13510
2019    13841
2020    13432
2021    17254
2022    20774
2023    26486
2024    28816


In [8]:
# ─────────────────────────────────────────────
# 5. CONFLICTOS MÁS FRECUENTES
# ─────────────────────────────────────────────
if 'conflict_name' in data.columns:
    top_conflicts = data['conflict_name'].value_counts().head(10)
    print("\n── Top 10 conflictos por número de eventos ─")
    for name, count in top_conflicts.items():
        print(f"   {name:<50} {count:>6,}")


── Top 10 conflictos por número de eventos ─
   Syria: Government                                  66,186
   Afghanistan: Government                            37,828
   Russia - Ukraine                                   28,499
   Turkey: Kurdistan                                   7,620
   Israel: Palestine                                   7,353
   India: Kashmir                                      7,237
   Syria: Islamic State                                6,695
   Iraq: Government                                    6,339
   IS - Civilians                                      5,943
   Colombia: Government                                5,891


In [9]:
# ─────────────────────────────────────────────
# 6. PRECISIÓN GEOGRÁFICA Y COBERTURA
# ─────────────────────────────────────────────
# where_prec: 1=exact, 2=within 25km, 3=within 100km, 4=capital, 5=country centroid
prec_labels = {
    1: 'Exacta',
    2: 'Dentro de 25 km',
    3: 'Dentro de 100 km',
    4: 'Capital del país',
    5: 'Centroide del país'
}

if 'where_prec' in data.columns:
    prec_dist = data['where_prec'].value_counts().sort_index()
    print("\n── Precisión geográfica de los eventos ───")
    for k, v in prec_dist.items():
        label = prec_labels.get(k, str(k))
        print(f"   {label:<25} {v:>8,}  ({v/len(data)*100:.1f}%)")
    exact_pct = (data['where_prec'] == 1).mean() * 100
    print(f"\n   → {exact_pct:.1f}% de los eventos tienen localización exacta")


── Precisión geográfica de los eventos ───
   Exacta                     179,745  (46.6%)
   Dentro de 25 km             92,676  (24.0%)
   Dentro de 100 km            55,381  (14.4%)
   Capital del país            36,336  (9.4%)
   Centroide del país          16,648  (4.3%)
   6                            4,881  (1.3%)
   7                              251  (0.1%)

   → 46.6% de los eventos tienen localización exacta


In [10]:
# ─────────────────────────────────────────────
# 7. ACTORES — los más activos (side_a / side_b)
# ─────────────────────────────────────────────
for side_col in ['side_a', 'side_b']:
    if side_col in data.columns:
        top_actors = data[side_col].value_counts().head(10)
        print(f"\n── Top 10 actores ({side_col}) ────────────────")
        for actor, count in top_actors.items():
            print(f"   {actor:<50} {count:>6,}")


── Top 10 actores (side_a) ────────────────
   Government of Syria                                75,181
   Government of Afghanistan                          40,223
   Government of Russia (Soviet Union)                32,460
   Government of India                                13,011
   IS                                                 12,257
   Government of Israel                                9,524
   Jalisco Cartel New Generation                       9,146
   Government of Myanmar (Burma)                       8,220
   Government of Turkey                                8,014
   Government of Bosnia-Herzegovina                    7,908

── Top 10 actores (side_b) ────────────────
   Syrian insurgents                                  66,186
   Civilians                                          59,605
   Taleban                                            35,935
   Government of Ukraine                              28,499
   IS                                                 21

In [11]:
# ─────────────────────────────────────────────
# 8. DESBALANCE DE VÍCTIMAS: civiles vs combatientes
# ─────────────────────────────────────────────
death_map = {
    'deaths_a'         : 'Bajas bando A',
    'deaths_b'         : 'Bajas bando B',
    'deaths_civilians' : 'Bajas civiles',
    'deaths_unknown'   : 'Bajas desconocidas',
}

present = {k: v for k, v in death_map.items() if k in data.columns}

if present:
    totals = {label: data[col].sum() for col, label in present.items()}
    grand  = sum(totals.values())
    print("\n── Distribución de bajas por categoría ───")
    for label, total in totals.items():
        pct = total / grand * 100 if grand else 0
        print(f"   {label:<25} {total:>10,.0f}  ({pct:.1f}%)")
    print(f"   {'TOTAL':<25} {grand:>10,.0f}")


── Distribución de bajas por categoría ───
   Bajas bando A                953,472  (24.1%)
   Bajas bando B                866,610  (21.9%)
   Bajas civiles              1,474,095  (37.3%)
   Bajas desconocidas           662,966  (16.8%)
   TOTAL                      3,957,143


In [12]:
# ─────────────────────────────────────────────
# 9. RESUMEN CARACTERÍSTICAS PRINCIPALES 
# ─────────────────────────────────────────────
print("\n══════════════════════════════════════════")
print("  RESUMEN — UCDP GED")
print("══════════════════════════════════════════")
print(f"  Alcance    : {num_rows:,} eventos · {num_cols} variables · {data['country'].nunique() if 'country' in data.columns else '?'} países")
print(f"  Período    : {year_range}")
if 'best' in data.columns:
    print(f"  Muertes est: {data['best'].sum():,.0f} (suma estimación central)")
if 'type_of_violence' in data.columns:
    most_common_type = type_labels.get(int(data['type_of_violence'].mode()[0]), 'N/D')
    print(f"  Tipo dom.  : {most_common_type}")
if 'country' in data.columns:
    print(f"  País más afectado: {data['country'].mode()[0]} ({data['country'].value_counts().iloc[0]:,} eventos)")
if 'conflict_name' in data.columns:
    print(f"  Conflicto más frecuente: {data['conflict_name'].mode()[0]}")
print()
print("  Variables clave:")
print("    Geográficas  : country, region, latitude, longitude, where_prec")
print("    Temporales   : year, date_start, date_end")
print("    Actores      : side_a, side_b, dyad_name, conflict_name")
print("    Víctimas     : best, low, high, deaths_a, deaths_b, deaths_civilians")
print("    Clasificación: type_of_violence, where_prec")
print("══════════════════════════════════════════")


══════════════════════════════════════════
  RESUMEN — UCDP GED
══════════════════════════════════════════
  Alcance    : 385,918 eventos · 49 variables · 124 países
  Período    : 1989 - 2024
  Muertes est: 3,957,143 (suma estimación central)
  Tipo dom.  : Violencia estatal
  País más afectado: Syria (87,861 eventos)
  Conflicto más frecuente: Syria: Government

  Variables clave:
    Geográficas  : country, region, latitude, longitude, where_prec
    Temporales   : year, date_start, date_end
    Actores      : side_a, side_b, dyad_name, conflict_name
    Víctimas     : best, low, high, deaths_a, deaths_b, deaths_civilians
    Clasificación: type_of_violence, where_prec
══════════════════════════════════════════
